# One-Rec — audio embeddings from 30s previews

Gives the recommender *ears*: for the top-60K most-playlisted tracks
(measured to cover ~67% of actually-served candidates), resolve a 30-second
preview clip (Deezer primary, iTunes fallback), embed it with **Essentia
Discogs-EffNet** (trained on 3M+ releases for music similarity), PCA to 256
dims, and export `audio_emb.parquet` for the `audio_cos_*` ranking features.

**Setup:** CPU session (no GPU needed), **Internet ON**, with the
`one-rec-train-ranker` kernel attached as a data source (provides
`track_meta.parquet`). Runtime ≈ 8–10 h; everything checkpoints every 2,000
tracks so a second session resumes safely.

Constraints honored: Deezer ≈10 req/s anonymous quota (errors arrive as JSON
`error.code=4` with HTTP 200); Deezer preview URLs expire in ~15 min, so
resolution and download are interleaved per batch and checkpoints store the
Deezer track *id*, never the URL.


In [ ]:
import asyncio, glob, io, json, os, re, sys, time, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

# Latest essentia-tensorflow (dev1438+) ships cp314-only wheels; Kaggle is
# Python 3.12 — pin the last cp312 build.
%pip install -q essentia-tensorflow==2.1b6.dev1389 aiohttp
!wget -q -nc https://essentia.upf.edu/models/feature-extractors/discogs-effnet/discogs-effnet-bs64-1.pb
print("model:", os.path.getsize("discogs-effnet-bs64-1.pb") / 1e6, "MB")


In [ ]:
CFG = dict(
    top_n=60_000,
    batch=500,            # resolve+download+embed unit; fits inside the 15-min URL TTL
    deezer_rps=8,         # 20% headroom under the 10 req/s quota
    itunes_interval=3.2,  # ~19 req/min, under the ~20/min documented limit
    itunes_cap=10_000,
    dur_tol_ms=7_000,     # kills most remasters/live/extended mismatches
    ckpt_every=2_000,
    embed_workers=4,      # one single-threaded TF per vCPU (see _init_worker)
    time_budget_h=10.0,   # stop cleanly BEFORE Kaggle's 12h kill so output persists
    pca_dim=256,
    seed=42,
)

WORK = Path("/kaggle/working")
hits = glob.glob("/kaggle/input/**/track_meta.parquet", recursive=True)
assert hits, "attach the one-rec-train-ranker kernel output as a data source"
meta = pd.read_parquet(hits[0])
top = meta.nlargest(CFG["top_n"], "playlist_count").reset_index(drop=True)
print(f"{len(meta):,} tracks in meta; targeting top {len(top):,} "
      f"(playlist_count >= {int(top['playlist_count'].min())})")

# ---- resume state (from this session AND any attached prior run's output) ----
done_ids = set()
emb_chunks = []
ckpt_files = sorted(WORK.glob("audio_ckpt_*.npz")) + sorted(
    Path(p) for p in glob.glob("/kaggle/input/**/audio_ckpt_*.npz", recursive=True)
)
for f in ckpt_files:
    z = np.load(f, allow_pickle=True)
    emb_chunks.append((z["ids"], z["embs"]))
    done_ids.update(z["ids"].tolist())

resolution_path = WORK / "preview_resolution.parquet"
resolution = []
for p in ([resolution_path] if resolution_path.exists() else []) + [
    Path(q) for q in glob.glob("/kaggle/input/**/preview_resolution.parquet", recursive=True)
]:
    resolution.extend(pd.read_parquet(p).to_dict("records"))
resolved_ids = {r["track_id"] for r in resolution}
print(f"resume: {len(done_ids):,} embedded, {len(resolved_ids):,} resolution records")


## Matching + resolution helpers

In [ ]:
VERSION_RE = re.compile(r"\b(live|karaoke|cover|tribute|acoustic|remix|instrumental)\b", re.I)


def norm(s):
    s = unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode().casefold()
    return re.sub(r"[^a-z0-9 ]+", " ", s).strip()


def toks(s):
    return set(norm(s).split())


def clean_title(name):
    name = re.sub(r"\(.*?\)|\[.*?\]", " ", name or "")
    return re.sub(r"\s*-\s*.*(remaster|version|edit|deluxe).*$", "", name, flags=re.I).strip()


def artist_ok(want, got):
    if norm(want) == norm(got):
        return True
    a, b = toks(want), toks(got)
    return len(a & b) / max(len(a | b), 1) >= 0.6


def pick_match(name, artist, duration_ms, candidates):
    """candidates: list of dicts with keys artist, title_version, duration_ms, ..."""
    for c in candidates:
        if not artist_ok(artist, c["artist"]):
            continue
        if duration_ms and c["duration_ms"] and abs(c["duration_ms"] - duration_ms) > CFG["dur_tol_ms"]:
            continue
        m = VERSION_RE.search(c.get("title_version") or "")
        if m and m.group(1).lower() not in norm(name):
            continue
        return c
    return None


class RateLimiter:
    def __init__(self, rps):
        self.min_int = 1.0 / rps
        self.next_t = 0.0
        self.lock = asyncio.Lock()

    async def acquire(self):
        async with self.lock:
            now = time.monotonic()
            wait = self.next_t - now
            self.next_t = max(now, self.next_t) + self.min_int
        if wait > 0:
            await asyncio.sleep(wait)


In [ ]:
import aiohttp

DEEZER_LIMITER = RateLimiter(CFG["deezer_rps"])


async def deezer_get(session, url, params=None, retries=5):
    for attempt in range(retries):
        await DEEZER_LIMITER.acquire()
        try:
            async with session.get(url, params=params, timeout=aiohttp.ClientTimeout(total=15)) as resp:
                data = await resp.json(content_type=None)
        except Exception:
            await asyncio.sleep(2 * (attempt + 1))
            continue
        # Quota errors come back as HTTP 200 with an error body.
        if isinstance(data, dict) and data.get("error", {}).get("code") == 4:
            await asyncio.sleep(5)
            continue
        return data
    return None


def _deezer_candidates(data):
    out = []
    for r in (data or {}).get("data", [])[:5]:
        out.append(dict(
            source="deezer", source_id=r.get("id"),
            artist=(r.get("artist") or {}).get("name", ""),
            title_version=r.get("title_version") or "",
            duration_ms=(r.get("duration") or 0) * 1000,
            preview=r.get("preview") or "",
        ))
    return out


async def resolve_deezer(session, row):
    for query, method in [(f"{row.artist} {row.name}", "plain"),
                          (f"{row.artist} {clean_title(row.name)}", "cleaned")]:
        data = await deezer_get(session, "https://api.deezer.com/search",
                                params={"q": query, "limit": "5"})
        match = pick_match(row.name, row.artist, row.duration_ms, _deezer_candidates(data))
        if match:
            match["method"] = method
            return match
    return None


async def fresh_deezer_preview(session, deezer_id):
    data = await deezer_get(session, f"https://api.deezer.com/track/{deezer_id}")
    return (data or {}).get("preview") or None


async def download(session, url, sem, source, source_id):
    async with sem:
        for attempt in range(3):
            try:
                async with session.get(url, timeout=aiohttp.ClientTimeout(total=30)) as resp:
                    if resp.status == 403 and source == "deezer":
                        url = await fresh_deezer_preview(session, source_id) or url
                        continue
                    if resp.status != 200:
                        await asyncio.sleep(1)
                        continue
                    return await resp.read()
            except Exception:
                await asyncio.sleep(1)
    return None


async def _resolve_and_download(rows):
    """One batch: resolve on Deezer, download hits immediately (inside the URL TTL)."""
    records, payloads = [], []
    sem = asyncio.Semaphore(12)
    async with aiohttp.ClientSession(headers={"User-Agent": "one-rec-research/1.0"}) as session:
        matches = await asyncio.gather(*(resolve_deezer(session, r) for r in rows.itertuples()))
        dl_tasks, dl_rows = [], []
        for row, match in zip(rows.itertuples(), matches):
            if match:
                records.append(dict(track_id=row.track_id, source="deezer",
                                    source_id=str(match["source_id"]), method=match["method"],
                                    duration_delta_ms=int(abs(match["duration_ms"] - row.duration_ms))))
                dl_tasks.append(download(session, match["preview"], sem, "deezer", match["source_id"]))
                dl_rows.append(row.track_id)
            else:
                records.append(dict(track_id=row.track_id, source="miss",
                                    source_id="", method="", duration_delta_ms=-1))
        audio = await asyncio.gather(*dl_tasks)
        payloads = [(tid, blob) for tid, blob in zip(dl_rows, audio) if blob]
    return records, payloads


def resolve_and_download(rows):
    return asyncio.run(_resolve_and_download(rows))


## iTunes fallback (background thread, ~19 req/min, stable preview URLs)

In [ ]:
import threading
import urllib.parse
import urllib.request

itunes_queue = []       # rows missed by Deezer (appended by the main loop)
itunes_resolved = []    # dicts with track_id + stable previewUrl
itunes_stop = threading.Event()


def itunes_worker():
    seen = 0
    while not itunes_stop.is_set() and seen < CFG["itunes_cap"]:
        try:
            row = itunes_queue.pop(0)
        except IndexError:
            time.sleep(5)
            continue
        seen += 1
        try:
            q = urllib.parse.urlencode({"term": f"{row.artist} {row.name}", "entity": "song", "limit": 5})
            with urllib.request.urlopen(f"https://itunes.apple.com/search?{q}", timeout=15) as resp:
                data = json.load(resp)
            cands = [dict(source="itunes", source_id=str(r.get("trackId", "")),
                          artist=r.get("artistName", ""), title_version="",
                          duration_ms=r.get("trackTimeMillis") or 0,
                          preview=r.get("previewUrl") or "")
                     for r in data.get("results", [])]
            match = pick_match(row.name, row.artist, row.duration_ms, [c for c in cands if c["preview"]])
            if match:
                itunes_resolved.append(dict(track_id=row.track_id, preview=match["preview"],
                                            source_id=match["source_id"],
                                            duration_delta_ms=int(abs(match["duration_ms"] - row.duration_ms))))
        except Exception:
            pass
        time.sleep(CFG["itunes_interval"])


threading.Thread(target=itunes_worker, daemon=True).start()
print("iTunes fallback thread started")


## Embedder — Discogs-EffNet in a fork pool (TF imported only in workers)

In [ ]:
import multiprocessing as mp
import tempfile

_model = None


def _init_worker():
    global _model, _MonoLoader
    # One single-threaded TF per worker: with defaults, each of the workers
    # spawns a full thread pool and they thrash the 4 vCPUs (~0.55 clips/s
    # total measured on the first run).
    os.environ["OMP_NUM_THREADS"] = "1"
    os.environ["TF_NUM_INTRAOP_THREADS"] = "1"
    os.environ["TF_NUM_INTEROP_THREADS"] = "1"
    from essentia.standard import MonoLoader, TensorflowPredictEffnetDiscogs
    _MonoLoader = MonoLoader
    _model = TensorflowPredictEffnetDiscogs(graphFilename="discogs-effnet-bs64-1.pb",
                                            output="PartitionedCall:1")


def _embed_one(args):
    tid, blob = args
    f = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False)
    try:
        f.write(blob)
        f.close()
        audio = _MonoLoader(filename=f.name, sampleRate=16000, resampleQuality=4)()
        if len(audio) < 16000:  # <1s of audio — corrupt download
            return tid, None
        emb = _model(audio).mean(axis=0).astype(np.float32)
        return tid, emb
    except Exception:
        return tid, None
    finally:
        os.unlink(f.name)


pool = mp.get_context("fork").Pool(CFG["embed_workers"], initializer=_init_worker)


def embed_batch(payloads):
    results = pool.map(_embed_one, payloads)
    return [(tid, emb) for tid, emb in results if emb is not None]


## Main pipeline — batches overlap: resolve/download batch N+1 while embedding batch N

In [ ]:
from concurrent.futures import ThreadPoolExecutor

# Resume rule: skip embedded tracks and known misses; tracks resolved but not
# yet embedded (crash between checkpoints) are simply re-resolved.
already_missed = {r["track_id"] for r in resolution if r["source"] == "miss"}
todo = top[~top["track_id"].isin(done_ids) & ~top["track_id"].isin(already_missed)].reset_index(drop=True)
print(f"{len(todo):,} tracks to process")

batches = [todo.iloc[i:i + CFG["batch"]] for i in range(0, len(todo), CFG["batch"])]
pending_ids, pending_embs = [], []
throughput_printed = False
t_start = time.time()

def flush_ckpt(force=False):
    global pending_ids, pending_embs, resolution
    if pending_ids and (force or len(pending_ids) >= CFG["ckpt_every"]):
        n = len(list(WORK.glob("audio_ckpt_*.npz")))
        np.savez(WORK / f"audio_ckpt_{n:03d}.npz",
                 ids=np.array(pending_ids, dtype=object),
                 embs=np.stack(pending_embs).astype(np.float16))
        emb_chunks.append((np.array(pending_ids, dtype=object),
                           np.stack(pending_embs).astype(np.float16)))
        pending_ids, pending_embs = [], []
    pd.DataFrame(resolution).to_parquet(resolution_path, index=False)


with ThreadPoolExecutor(max_workers=1) as fetcher:
    future = fetcher.submit(resolve_and_download, batches[0]) if batches else None
    for bi in range(len(batches)):
        records, payloads = future.result()
        if bi + 1 < len(batches):
            future = fetcher.submit(resolve_and_download, batches[bi + 1])

        resolution.extend(records)
        by_id = {r.track_id: r for r in batches[bi].itertuples()}
        for rec in records:
            if rec["source"] == "miss" and len(itunes_queue) < CFG["itunes_cap"]:
                itunes_queue.append(by_id[rec["track_id"]])

        t0 = time.time()
        embedded = embed_batch(payloads)
        for tid, emb in embedded:
            pending_ids.append(tid)
            pending_embs.append(emb)
        if not throughput_printed and embedded:
            rate = len(payloads) / max(time.time() - t0, 1e-9)
            print(f"embed throughput: {rate:.2f} clips/s "
                  f"({'OK' if rate >= 1.5 else 'SLOW — expect a second session; checkpoints cover it'})")
            throughput_printed = True

        flush_ckpt()
        elapsed = (time.time() - t_start) / 3600
        if bi % 10 == 0:
            done_total = sum(len(c[0]) for c in emb_chunks) + len(pending_ids)
            print(f"batch {bi+1}/{len(batches)} | embedded {done_total:,} | "
                  f"misses queued {len(itunes_queue):,} | itunes hits {len(itunes_resolved):,} | "
                  f"{elapsed:.1f}h", flush=True)
        if elapsed > CFG["time_budget_h"]:
            # Stop cleanly so this version COMPLETES and its checkpoints persist
            # as attachable output; the next run resumes from them.
            print(f"time budget {CFG['time_budget_h']}h reached at batch {bi+1} — stopping cleanly")
            break

flush_ckpt(force=True)
print("main loop done")


## iTunes leftovers — stable URLs, so download+embed at the end

In [ ]:
async def _download_itunes(entries):
    sem = asyncio.Semaphore(12)
    async with aiohttp.ClientSession() as session:
        blobs = await asyncio.gather(*(download(session, e["preview"], sem, "itunes", e["source_id"])
                                       for e in entries))
    return [(e["track_id"], b) for e, b in zip(entries, blobs) if b]

itunes_stop.set()
already = done_ids | set(pending_ids) | {tid for ids, _ in emb_chunks for tid in ids}
entries = [e for e in itunes_resolved if e["track_id"] not in already]
print(f"{len(entries):,} iTunes-resolved tracks to embed")
for i in range(0, len(entries), CFG["batch"]):
    chunk = entries[i:i + CFG["batch"]]
    payloads = asyncio.run(_download_itunes(chunk))
    for tid, emb in embed_batch(payloads):
        pending_ids.append(tid)
        pending_embs.append(emb)
    for e in chunk:
        resolution.append(dict(track_id=e["track_id"], source="itunes",
                               source_id=e["source_id"], method="itunes",
                               duration_delta_ms=e["duration_delta_ms"]))
    flush_ckpt()
flush_ckpt(force=True)
pool.close()


## Export — PCA→256, fp16 parquet, audit sample

In [ ]:
from sklearn.decomposition import PCA

all_ids = np.concatenate([ids for ids, _ in emb_chunks]) if emb_chunks else np.array([], dtype=object)
all_embs = (np.concatenate([e.astype(np.float32) for _, e in emb_chunks])
            if emb_chunks else np.zeros((0, 1280), np.float32))
# Dedup (session resumes can overlap a checkpoint boundary)
_, keep = np.unique(all_ids, return_index=True)
all_ids, all_embs = all_ids[np.sort(keep)], all_embs[np.sort(keep)]
print(f"embedded {len(all_ids):,}/{CFG['top_n']:,} tracks "
      f"({len(all_ids)/CFG['top_n']:.1%} of target)")

pca = PCA(n_components=CFG["pca_dim"], random_state=CFG["seed"]).fit(all_embs)
reduced = pca.transform(all_embs).astype(np.float16)
print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.1%}")

pd.DataFrame({"track_id": all_ids,
              "embedding": [row.tobytes() for row in reduced]}
             ).to_parquet(WORK / "audio_emb.parquet", index=False)
pd.DataFrame({"track_id": all_ids,
              "embedding": [row.astype(np.float16).tobytes() for row in all_embs]}
             ).to_parquet(WORK / "audio_emb_full_fp16.parquet", index=False)
np.savez(WORK / "pca_audio.npz", mean=pca.mean_.astype(np.float32),
         components=pca.components_.astype(np.float32))
pd.DataFrame(resolution).drop_duplicates("track_id").to_parquet(resolution_path, index=False)

res_df = pd.read_parquet(resolution_path)
print(res_df["source"].value_counts().to_string())
print("\naudit sample (check these matched the right versions):")
sample = res_df[res_df.source != "miss"].sample(min(100, (res_df.source != "miss").sum()),
                                                random_state=1)
print(sample.merge(top[["track_id", "name", "artist"]], on="track_id")
      [["name", "artist", "source", "method", "duration_delta_ms"]].head(30).to_string())

print("\nfiles:", [f.name for f in WORK.glob('*.parquet')] + ["pca_audio.npz"])
print(f"audio_emb.parquet: {(WORK / 'audio_emb.parquet').stat().st_size / 1e6:.1f} MB")
